# Cross-hazard sector EAD panels

This notebook creates comparable sector-level stacked bar charts for coastal flood, river flood, and landslide avoided-EAD results.

The plotted convention is:

- full grey bar = EAD
- hatched green segment = avoided EAD
- visible grey lower segment = residual or with-nature EAD

Outputs are saved as individual hazard panels and as four-panel figures for the minimum and maximum scenarios.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.ticker import FuncFormatter, MultipleLocator


In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis")
paper_root = base_path / "dphil_papers"
output_dir = paper_root / "dphil_paper_3/results/03_cross_hazard_comparison/hazard_sector_ead_panels"
output_dir.mkdir(parents=True, exist_ok=True)

sector_order = ["buildings", "transport", "water", "energy", "TOTAL"]
sector_labels = {
    "buildings": "buildings",
    "transport": "transport",
    "water": "water",
    "energy": "energy",
    "TOTAL": "TOTAL",
}
scenario_order = ["minimum", "maximum"]
hazard_order = ["Coastal flood", "River flood", "Landslide"]

jmd_per_usd = 150.0
usd_per_jmd = 1.0 / jmd_per_usd
millions = 1e6
plot_percent_labels = True

river_scenario_ids = {
    "minimum": "10",
    "maximum": "7",
}
river_sector_map = {
    "buildings_assigned_economic_activity_areas": "buildings",
    "rail_edges": "transport",
    "rail_nodes": "transport",
    "roads_edges": "transport",
    "roads_nodes": "transport",
    "airport_polygon_areas": "transport",
    "port_polygon_areas": "transport",
    "irrigation_assets_NIC_edges": "water",
    "irrigation_assets_NIC_nodes": "water",
    "pipelines_NWC_edges": "water",
    "potable_facilities_NWC_nodes": "water",
    "waste_water_facilities_NWC_nodes": "water",
    "electricity_network_v3.1_nodes": "energy",
}


In [ ]:
def add_total_sector_row(sector_data):
    total_row = pd.DataFrame(
        [
            {
                "sector": "TOTAL",
                "counterfactual_ead_usd": sector_data["counterfactual_ead_usd"].sum(),
                "residual_ead_usd": sector_data["residual_ead_usd"].sum(),
                "avoided_ead_usd": sector_data["avoided_ead_usd"].sum(),
            }
        ]
    )
    return pd.concat([sector_data, total_row], ignore_index=True)


def add_million_columns(sector_data):
    sector_data = sector_data.copy()
    sector_data["counterfactual_ead_usd_mn"] = sector_data["counterfactual_ead_usd"] / millions
    sector_data["residual_ead_usd_mn"] = sector_data["residual_ead_usd"] / millions
    sector_data["avoided_ead_usd_mn"] = sector_data["avoided_ead_usd"] / millions
    sector_data["avoided_share_pct"] = np.where(
        sector_data["counterfactual_ead_usd"] > 0,
        100.0 * sector_data["avoided_ead_usd"] / sector_data["counterfactual_ead_usd"],
        np.nan,
    )
    return sector_data


def complete_sector_order(sector_data):
    sector_data = sector_data.set_index("sector").reindex(sector_order).fillna(0).reset_index()
    return sector_data


def load_coastal_sector_data():
    coastal_paths = {
        "minimum": paper_root
        / "dphil_paper_3/results/02_damage_estimates/coastal_flood_damages/results_coastal_minimum_scenario/damage_estimates/coastal_ead_sector_summary_usd_with_pct_avoided.csv",
        "maximum": paper_root
        / "dphil_paper_3/results/02_damage_estimates/coastal_flood_damages/results_coastal_maximum_scenario/damage_estimates/coastal_ead_sector_summary_usd_with_pct_avoided.csv",
    }
    coastal_tables = []

    for scenario_name, coastal_path in coastal_paths.items():
        raw_coastal_data = pd.read_csv(coastal_path)
        sector_data = pd.DataFrame(
            {
                "sector": raw_coastal_data["Sector"].str.lower(),
                "counterfactual_ead_usd": raw_coastal_data["EAD_Without_Mangroves_USD"],
                "residual_ead_usd": raw_coastal_data["EAD_With_Mangroves_USD"],
                "avoided_ead_usd": raw_coastal_data["Avoided_EAD_USD"],
            }
        )
        sector_data = add_total_sector_row(sector_data)
        sector_data = complete_sector_order(sector_data)
        sector_data["hazard"] = "Coastal flood"
        sector_data["scenario"] = scenario_name
        sector_data["variant"] = "mangrove protection"
        sector_data["counterfactual_label"] = "Without mangroves EAD"
        sector_data["avoided_label"] = "Avoided by mangroves"
        coastal_tables.append(sector_data)

    return pd.concat(coastal_tables, ignore_index=True)


def load_landslide_sector_data():
    landslide_paths = {
        "minimum": paper_root
        / "dphil_paper_3/results/02_damage_estimates/landslide_damages/results_landslide_minimum_scenario_combined_class/damage_estimates/landslide_ead_avoided_sector_totals_usd_combined_class.csv",
        "maximum": paper_root
        / "dphil_paper_3/results/02_damage_estimates/landslide_damages/results_landslide_maximum_scenario_combined_class/damage_estimates/landslide_ead_avoided_sector_totals_usd_combined_class.csv",
    }
    landslide_variant_columns = {
        "protection": {
            "counterfactual_column": "EAD_Deforestation_USD",
            "residual_column": "EAD_Baseline_USD",
            "avoided_column": "Avoided_EAD_Protection_USD",
            "counterfactual_label": "Deforestation EAD",
            "avoided_label": "Avoided by forest protection",
        },
        "forest restoration": {
            "counterfactual_column": "EAD_Baseline_USD",
            "residual_column": "EAD_Reafforestation_USD",
            "avoided_column": "Avoided_EAD_Reafforestation_USD",
            "counterfactual_label": "Baseline EAD",
            "avoided_label": "Avoided by forest restoration",
        },
    }
    landslide_tables = []

    for scenario_name, landslide_path in landslide_paths.items():
        raw_landslide_data = pd.read_csv(landslide_path)

        for variant_name, variant_columns in landslide_variant_columns.items():
            sector_data = pd.DataFrame(
                {
                    "sector": raw_landslide_data["Sector"].str.lower(),
                    "counterfactual_ead_usd": raw_landslide_data[variant_columns["counterfactual_column"]],
                    "residual_ead_usd": raw_landslide_data[variant_columns["residual_column"]],
                    "avoided_ead_usd": raw_landslide_data[variant_columns["avoided_column"]],
                }
            )
            sector_data = add_total_sector_row(sector_data)
            sector_data = complete_sector_order(sector_data)
            sector_data["hazard"] = "Landslide"
            sector_data["scenario"] = scenario_name
            sector_data["variant"] = variant_name
            sector_data["counterfactual_label"] = variant_columns["counterfactual_label"]
            sector_data["avoided_label"] = variant_columns["avoided_label"]
            landslide_tables.append(sector_data)

    return pd.concat(landslide_tables, ignore_index=True)


def load_river_sector_data():
    river_damage_path = paper_root / "dphil_paper_2/processed_data/nbs_river_catchment/damage_future/damage__future.parquet"
    raw_river_data = pd.read_parquet(river_damage_path)
    raw_river_data["ensemble_member"] = raw_river_data["ensemble_member"].astype(str)
    river_tables = []

    required_columns = {"asset_class", "baseline__fluvial__ead", "future__fluvial__ead", "ensemble_member"}
    missing_columns = required_columns - set(raw_river_data.columns)
    if missing_columns:
        raise KeyError(f"Missing river damage columns: {sorted(missing_columns)}")

    for scenario_name, ensemble_member_id in river_scenario_ids.items():
        scenario_data = raw_river_data.loc[raw_river_data["ensemble_member"] == ensemble_member_id].copy()
        scenario_data["baseline__fluvial__ead"] = pd.to_numeric(
            scenario_data["baseline__fluvial__ead"], errors="coerce"
        )
        scenario_data["future__fluvial__ead"] = pd.to_numeric(
            scenario_data["future__fluvial__ead"], errors="coerce"
        )
        scenario_data["avoided_ead_jmd"] = (
            scenario_data["baseline__fluvial__ead"] - scenario_data["future__fluvial__ead"]
        )
        scenario_data["sector"] = scenario_data["asset_class"].map(river_sector_map)
        unclassified_asset_classes = sorted(scenario_data.loc[scenario_data["sector"].isna(), "asset_class"].dropna().unique())
        if unclassified_asset_classes:
            raise ValueError(f"Unclassified river asset classes: {unclassified_asset_classes}")

        sector_data = (
            scenario_data.groupby("sector", as_index=False)
            .agg(
                counterfactual_ead_jmd=("baseline__fluvial__ead", "sum"),
                residual_ead_jmd=("future__fluvial__ead", "sum"),
                avoided_ead_jmd=("avoided_ead_jmd", "sum"),
            )
            .assign(
                counterfactual_ead_usd=lambda data_frame: data_frame["counterfactual_ead_jmd"] * usd_per_jmd,
                residual_ead_usd=lambda data_frame: data_frame["residual_ead_jmd"] * usd_per_jmd,
                avoided_ead_usd=lambda data_frame: data_frame["avoided_ead_jmd"] * usd_per_jmd,
            )
        )
        sector_data = sector_data[["sector", "counterfactual_ead_usd", "residual_ead_usd", "avoided_ead_usd"]]
        sector_data = add_total_sector_row(sector_data)
        sector_data = complete_sector_order(sector_data)
        sector_data["hazard"] = "River flood"
        sector_data["scenario"] = scenario_name
        sector_data["variant"] = "forest restoration"
        sector_data["counterfactual_label"] = "Baseline EAD"
        sector_data["avoided_label"] = "Avoided through forest restoration"
        river_tables.append(sector_data)

    return pd.concat(river_tables, ignore_index=True)


def build_plot_data():
    plot_data = pd.concat(
        [
            load_coastal_sector_data(),
            load_river_sector_data(),
            load_landslide_sector_data(),
        ],
        ignore_index=True,
    )
    plot_data = add_million_columns(plot_data)
    plot_data["hazard"] = pd.Categorical(plot_data["hazard"], categories=hazard_order, ordered=True)
    plot_data["scenario"] = pd.Categorical(plot_data["scenario"], categories=scenario_order, ordered=True)
    plot_data["sector"] = pd.Categorical(plot_data["sector"], categories=sector_order, ordered=True)
    return plot_data.sort_values(["hazard", "variant", "scenario", "sector"]).reset_index(drop=True)


In [ ]:
def slugify_label(label):
    return str(label).lower().replace(" ", "_").replace("/", "_").replace("-", "_")


def choose_y_axis_interval(maximum_value):
    if maximum_value <= 50:
        return 10
    if maximum_value <= 100:
        return 20
    if maximum_value <= 500:
        return 100
    if maximum_value <= 1000:
        return 200
    if maximum_value <= 5000:
        return 1000
    return 2000


def draw_stacked_ead_bars(panel_axis, panel_data, title, show_y_label=True):
    ordered_panel_data = panel_data.set_index("sector").loc[sector_order].reset_index()
    x_positions = np.arange(len(ordered_panel_data))
    is_total_sector = ordered_panel_data["sector"].eq("TOTAL").to_numpy()
    counterfactual_values = ordered_panel_data["counterfactual_ead_usd_mn"].to_numpy()
    avoided_values = np.minimum(ordered_panel_data["avoided_ead_usd_mn"].to_numpy(), counterfactual_values)
    residual_values = np.maximum(counterfactual_values - avoided_values, 0)
    base_colors = np.where(is_total_sector, "#B0B0B0", "#C8C8C8")

    panel_axis.bar(
        x_positions,
        counterfactual_values,
        color=base_colors,
        edgecolor="#333333",
        linewidth=0.6,
        label="EAD",
    )
    panel_axis.bar(
        x_positions,
        avoided_values,
        bottom=residual_values,
        facecolor="none",
        edgecolor="#2E7D32",
        linewidth=0.9,
        hatch="///",
        label="Avoided EAD",
        zorder=3,
    )

    if len(ordered_panel_data) > 1:
        panel_axis.axvline(len(ordered_panel_data) - 1.5, linestyle=":", color="0.5", linewidth=0.8)

    if plot_percent_labels:
        y_axis_minimum, y_axis_maximum = panel_axis.get_ylim()
        label_offset = 0.02 * (y_axis_maximum - y_axis_minimum)
        for row_index, row_data in ordered_panel_data.iterrows():
            if row_data["counterfactual_ead_usd_mn"] <= 0 or row_data["avoided_ead_usd_mn"] <= 0:
                continue
            label_y_position = row_data["counterfactual_ead_usd_mn"] + label_offset
            panel_axis.text(
                x_positions[row_index],
                label_y_position,
                f"{row_data['avoided_share_pct']:.1f}%",
                ha="center",
                va="bottom",
                fontsize=8,
                color="#2E7D32",
                bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.8, "pad": 1.0},
                clip_on=False,
                zorder=5,
            )

    panel_axis.grid(axis="y", linestyle=":", color="0.88", zorder=0)
    panel_axis.set_axisbelow(True)
    panel_axis.set_xticks(x_positions)
    panel_axis.set_xticklabels(
        [sector_labels[sector_name] for sector_name in ordered_panel_data["sector"]],
        rotation=20,
        ha="right",
        fontsize=9,
    )
    panel_axis.set_title(title, fontsize=11, fontweight="bold")
    panel_axis.yaxis.set_major_formatter(FuncFormatter(lambda tick_value, tick_position: f"{tick_value:,.0f}"))
    panel_axis.yaxis.set_major_locator(MultipleLocator(choose_y_axis_interval(float(np.nanmax(counterfactual_values)))))
    panel_axis.margins(y=0.08)

    if show_y_label:
        panel_axis.set_ylabel("EAD (US$ million)")


def save_figure_versions(figure, output_file_stem):
    saved_paths = []
    for output_suffix in [".png", ".svg", ".pdf"]:
        output_path = output_file_stem.with_suffix(output_suffix)
        figure.savefig(output_path, dpi=600, bbox_inches="tight", facecolor="white")
        saved_paths.append(output_path)
    return saved_paths


def save_single_panel_figure(panel_data, output_file_stem):
    hazard_name = str(panel_data["hazard"].iloc[0])
    scenario_name = str(panel_data["scenario"].iloc[0])
    variant_name = str(panel_data["variant"].iloc[0])
    title = f"{hazard_name}: {variant_name}, {scenario_name} scenario"
    figure, panel_axis = plt.subplots(figsize=(8.5, 4.8))
    draw_stacked_ead_bars(panel_axis, panel_data, title, show_y_label=True)
    panel_axis.legend(
        frameon=False,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.14),
        ncol=2,
        borderaxespad=0.0,
    )
    figure.subplots_adjust(bottom=0.25, right=0.98)
    saved_paths = save_figure_versions(figure, output_file_stem)
    plt.close(figure)
    return saved_paths


def save_four_hazard_figure(plot_data, scenario_name, output_file_stem, display_inline=True):
    panel_specs = [
        {"hazard": "Coastal flood", "variant": None, "title": "Coastal flood: mangrove protection"},
        {"hazard": "River flood", "variant": None, "title": "River flood: forest restoration"},
        {"hazard": "Landslide", "variant": "protection", "title": "Landslide: forest protection"},
        {"hazard": "Landslide", "variant": "forest restoration", "title": "Landslide: forest restoration"},
    ]
    figure, panel_axes = plt.subplots(2, 2, figsize=(13.5, 9.0), sharey=False)

    for panel_index, (panel_axis, panel_spec) in enumerate(zip(panel_axes.ravel(), panel_specs)):
        panel_data = plot_data.loc[
            (plot_data["hazard"] == panel_spec["hazard"])
            & (plot_data["scenario"] == scenario_name)
        ]
        if panel_spec["variant"] is not None:
            panel_data = panel_data.loc[panel_data["variant"] == panel_spec["variant"]]

        draw_stacked_ead_bars(
            panel_axis,
            panel_data,
            panel_spec["title"],
            show_y_label=panel_index % 2 == 0,
        )

    landslide_panel_data = plot_data.loc[
        (plot_data["hazard"] == "Landslide")
        & (plot_data["scenario"] == scenario_name)
        & (plot_data["variant"].isin(["protection", "forest restoration"]))
    ]
    landslide_maximum_ead = float(landslide_panel_data["counterfactual_ead_usd_mn"].max())
    landslide_tick_interval = choose_y_axis_interval(landslide_maximum_ead)
    landslide_axis_maximum = np.ceil(landslide_maximum_ead / landslide_tick_interval) * landslide_tick_interval
    for landslide_axis in panel_axes[1, :]:
        landslide_axis.set_ylim(0, landslide_axis_maximum)
        landslide_axis.yaxis.set_major_locator(MultipleLocator(landslide_tick_interval))

    legend_handles, legend_labels = panel_axes.ravel()[0].get_legend_handles_labels()
    figure.legend(
        legend_handles,
        legend_labels,
        frameon=False,
        loc="lower center",
        bbox_to_anchor=(0.5, -0.01),
        ncol=2,
    )
    figure.suptitle(f"Sector EAD and avoided EAD by hazard, {scenario_name} scenario", fontweight="bold", y=0.99)
    figure.subplots_adjust(bottom=0.12, top=0.92, wspace=0.25, hspace=0.42)
    saved_paths = save_figure_versions(figure, output_file_stem)
    if display_inline:
        display(figure)
    plt.close(figure)
    return saved_paths


In [ ]:
plot_data = build_plot_data()
plot_data_path = output_dir / "cross_hazard_sector_ead_plot_data.csv"
plot_data.to_csv(plot_data_path, index=False)

display_columns = [
    "hazard",
    "variant",
    "scenario",
    "sector",
    "counterfactual_ead_usd_mn",
    "residual_ead_usd_mn",
    "avoided_ead_usd_mn",
    "avoided_share_pct",
]
display(plot_data[display_columns].round(2))
print(f"Saved plot data to: {plot_data_path}")


In [ ]:
saved_figure_paths = []

for (hazard_name, variant_name, scenario_name), panel_data in plot_data.groupby(
    ["hazard", "variant", "scenario"], observed=True
):
    output_file_stem = output_dir / (
        f"{slugify_label(hazard_name)}_{slugify_label(variant_name)}_{scenario_name}_sector_ead_panel"
    )
    saved_figure_paths.extend(save_single_panel_figure(panel_data, output_file_stem))

for scenario_name in scenario_order:
    output_file_stem = output_dir / f"four_hazard_sector_ead_panel_{scenario_name}"
    saved_figure_paths.extend(save_four_hazard_figure(plot_data, scenario_name, output_file_stem))

saved_figures = pd.DataFrame({"figure_path": [str(path) for path in saved_figure_paths]})
saved_figures_path = output_dir / "cross_hazard_sector_ead_saved_figures.csv"
saved_figures.to_csv(saved_figures_path, index=False)

display(saved_figures)
print(f"Saved figure list to: {saved_figures_path}")
